In [1]:
import requests
import pandas as pd

url_prefix = 'https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/03-evaluation/'
docs_url = url_prefix + 'search_evaluation/documents-with-ids.json'
documents = requests.get(docs_url).json()

ground_truth_url = url_prefix + 'search_evaluation/ground-truth-data.csv'
df_ground_truth = pd.read_csv(ground_truth_url)
ground_truth = df_ground_truth.to_dict(orient='records')

In [2]:
from tqdm.auto import tqdm

def hit_rate(relevance_total):
    cnt = 0

    for line in relevance_total:
        if True in line:
            cnt = cnt + 1

    return cnt / len(relevance_total)

def mrr(relevance_total):
    total_score = 0.0

    for line in relevance_total:
        for rank in range(len(line)):
            if line[rank] == True:
                total_score = total_score + 1 / (rank + 1)

    return total_score / len(relevance_total)

def evaluate(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        doc_id = q['document']
        results = search_function(q)
        relevance = [d['id'] == doc_id for d in results]
        relevance_total.append(relevance)

    return {
        'hit_rate': hit_rate(relevance_total),
        'mrr': mrr(relevance_total),
    }

/Users/dayujiang/opt/anaconda3/envs/llm/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import minsearch

index = minsearch.Index(
    text_fields=["question", "text", "section"],
    keyword_fields=["course", "id"]
)

index.fit(documents)

In [4]:
def minsearch_search(query, course):
    boost = {'question': 1.5, 'section': 0.1}

    results = index.search(
        query=query,
        filter_dict={'course': course},
        boost_dict=boost,
        num_results=5
    )

    return results

In [5]:
relevance_total = []

for q in tqdm(ground_truth):
    doc_id = q['document']
    results = minsearch_search(query=q['question'], course=q['course'])
    relevance = [d['id'] == doc_id for d in results]
    relevance_total.append(relevance)

print("Q1: What's the hitrate for this approach?")
hit_rate(relevance_total)

100%|██████████| 4627/4627 [00:21<00:00, 217.56it/s]

Q1: What's the hitrate for this approach?


0.848714069591528

In [6]:
from minsearch import VectorSearch

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.pipeline import make_pipeline

In [7]:
texts = []

for doc in documents:
    t = doc['question']
    texts.append(t)

pipeline = make_pipeline(
    TfidfVectorizer(min_df=3),
    TruncatedSVD(n_components=128, random_state=1)
)
X = pipeline.fit_transform(texts)

In [8]:
vindex = VectorSearch(keyword_fields={'course'})
vindex.fit(X, documents)

In [9]:
def vector_search(query_vector, course):
    results = vindex.search(
        query_vector=query_vector,
        filter_dict={'course': course},
        num_results=5
    )

    return results

In [10]:
relevance_total = []

for q in tqdm(ground_truth):
    doc_id = q['document']
    query_vector = pipeline.transform([q['question']])
    results = vector_search(query_vector=query_vector[0], course=q['course'])
    relevance = [d['id'] == doc_id for d in results]
    relevance_total.append(relevance)

print("Q2:  What's MRR for it?")
mrr(relevance_total)

100%|██████████| 4627/4627 [00:10<00:00, 440.21it/s]

Q2:  What's MRR for it?


0.3572833369353793

In [11]:
texts = []

for doc in documents:
    t = doc['question'] + ' ' + doc['text']
    texts.append(t)

X = pipeline.fit_transform(texts)
vindex.fit(X, documents)

In [12]:
relevance_total = []

for q in tqdm(ground_truth):
    doc_id = q['document']
    query_vector = pipeline.transform([q['question']])
    results = vector_search(query_vector=query_vector[0], course=q['course'])
    relevance = [d['id'] == doc_id for d in results]
    relevance_total.append(relevance)

print("Q3:  What's the hitrate?")
hit_rate(relevance_total)

100%|██████████| 4627/4627 [00:10<00:00, 461.65it/s]

Q3:  What's the hitrate?


0.8210503566025502

In [13]:
# Complete working solution
from qdrant_client import QdrantClient, models

client = QdrantClient("http://localhost:6333")
collection_name = "hw3"
EMBEDDING_DIMENSIONALITY = 512
model_handle = "jinaai/jina-embeddings-v2-small-en"

# Delete and recreate collection
try:
    client.delete_collection(collection_name)
except:
    pass

client.create_collection(
    collection_name=collection_name,
    vectors_config=models.VectorParams(
        size=EMBEDDING_DIMENSIONALITY,
        distance=models.Distance.COSINE
    )
)

True

In [14]:
# Create points with integer IDs and original IDs in payload
points = []
for i, doc in enumerate(documents):
    point = models.PointStruct(
        id=i,  # Integer ID for Qdrant
        vector=models.Document(text=doc['question'] + ' ' + doc['text'], model=model_handle),
        payload={
            "text": doc['text'],
            "section": doc['section'],
            "course": doc['course'],
            "original_id": doc['id']  # Store original ID here
        }
    )
    points.append(point)

client.upsert(collection_name=collection_name, points=points)

UpdateResult(operation_id=0, status=<UpdateStatus.COMPLETED: 'completed'>)

In [15]:
def qdrant_search(query, limit=5):

    results = client.query_points(
        collection_name=collection_name,
        query=models.Document( #embed the query text locally with "jinaai/jina-embeddings-v2-small-en"
            text=query,
            model=model_handle 
        ),
        limit=limit, # top closest matches
        with_payload=True #to get metadata in the results
    )

    return results



In [16]:
relevance_total = []

for q in tqdm(ground_truth):
    doc_id = q['document']
    results = qdrant_search(query=q['question'])
    
    # QueryResponse has a .points attribute
    points = results.points
    
    relevance = [point.payload.get('original_id') == doc_id for point in points]
    relevance_total.append(relevance)

print("Q5:  What's MRR for it?")
mrr(relevance_total)

100%|██████████| 4627/4627 [01:02<00:00, 73.76it/s]

Q5:  What's MRR for it?


0.8244362798069315

In [17]:
import numpy as np

def cosine(u, v):
    u_norm = np.sqrt(u.dot(u))
    v_norm = np.sqrt(v.dot(v))
    return u.dot(v) / (u_norm * v_norm)


In [18]:
results_url = url_prefix + 'rag_evaluation/data/results-gpt4o-mini.csv'
df_results = pd.read_csv(results_url)

pipeline = make_pipeline(
    TfidfVectorizer(min_df=3),
    TruncatedSVD(n_components=128, random_state=1)
)

pipeline.fit(df_results.answer_llm + ' ' + df_results.answer_orig + ' ' + df_results.question)

Pipeline(steps=[('tfidfvectorizer', TfidfVectorizer(min_df=3)),
                ('truncatedsvd',
                 TruncatedSVD(n_components=128, random_state=1))])

In [19]:
# Calculate cosine similarity for each pair
cosine_similarities = []

for idx, row in df_results.iterrows():
    # Get embeddings for LLM answer and original answer
    v_llm = pipeline.transform([row['answer_llm']])[0]
    v_orig = pipeline.transform([row['answer_orig']])[0]
    
    # Calculate cosine similarity
    similarity = cosine(v_llm, v_orig)
    cosine_similarities.append(similarity)

# Calculate average cosine similarity
average_cosine = np.mean(cosine_similarities)

print("Q5: What's the average cosine? " + str(average_cosine))

Q5: What's the average cosine? 0.8415841233490402


In [20]:
from rouge_score import rouge_scorer

# Create the scorer
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

# Test with your data
r = df_results.iloc[10]
scores = scorer.score(r['answer_orig'], r['answer_llm'])
print(scores)

{'rouge1': Score(precision=0.5, recall=0.5128205128205128, fmeasure=0.5063291139240506), 'rouge2': Score(precision=0.3076923076923077, recall=0.3157894736842105, fmeasure=0.3116883116883117), 'rougeL': Score(precision=0.375, recall=0.38461538461538464, fmeasure=0.37974683544303806)}


In [22]:
# Now calculate ROUGE-1 F1 for all pairs in the dataframe
rouge1_f1_scores = []

for idx, row in df_results.iterrows():
    # Calculate ROUGE scores
    scores = scorer.score(row['answer_orig'], row['answer_llm'])
    
    # Extract ROUGE-1 F1 score
    rouge1_f1 = scores['rouge1'].fmeasure
    rouge1_f1_scores.append(rouge1_f1)

# Calculate average ROUGE-1 F1 score
average_rouge1_f1 = sum(rouge1_f1_scores) / len(rouge1_f1_scores)

print(f"Q6: What's the average Rouge-1 F1?{average_rouge1_f1}")

Q6: What's the average Rouge-1 F1?0.451029508922461
